# 005 · Printing and Input

> **Syntax › Basics** · Python 3.14

`print()` and `input()` are Python's built-in equivalents of `Console.WriteLine` and `Console.ReadLine`. `print()` in particular is far more flexible than it looks, thanks to a few keyword arguments.

**In this notebook you will learn:**

- how `print()` maps to `Console.WriteLine` / `Console.Write`
- controlling output with `sep`, `end`, `file` and `flush`
- the basics of f-strings for building output
- reading user input with `input()` and converting it with `int()` / `float()`
- pretty-printing nested data with `pprint`

## C# mental model

| C# | Python |
|---|---|
| `Console.WriteLine(x)` | `print(x)` |
| `Console.Write(x)` (no newline) | `print(x, end="")` |
| `Console.WriteLine($"Hi {name}")` | `print(f"Hi {name}")` |
| `Console.Error.WriteLine(x)` | `print(x, file=sys.stderr)` |
| `Console.Out.Flush()` | `print(x, flush=True)` or `sys.stdout.flush()` |
| `Console.ReadLine()` | `input()` |
| `int.Parse(s)` | `int(s)` |

## `print()` basics

`print()` is a regular built-in function (it was a statement in Python 2 — you may still see `print "x"` in very old code). It converts each argument to a string with `str()` and writes it to standard output followed by a newline.

```csharp
// C#
Console.WriteLine("Hello, world!");
Console.WriteLine(42);
Console.WriteLine();          // empty line
```

In [ ]:
print("Hello, world!")
print(42)
print()                       # empty line
print([1, 2, 3], {"a": 1}, None, True)   # any object can be printed

## Multiple arguments and `sep`

Unlike `Console.WriteLine`, `print()` accepts **any number of arguments** and joins them with a separator — a single space by default. Change it with `sep=`.

```csharp
// C#
Console.WriteLine(string.Join(" ", "alice", 30, "admin"));
Console.WriteLine(string.Join(", ", "alice", 30, "admin"));
```

In [ ]:
print("alice", 30, "admin")
print("alice", 30, "admin", sep=", ")
print("2026", "10", "12", sep="-")
print("a", "b", "c", sep="")         # no separator at all

⚠️ **Gotcha:** `print("Total:", 5)` adds a space for you. Don't also add one inside the string, or you'll get two: `print("Total: ", 5)` → `Total:  5`.

## `end` — the `Console.Write` equivalent

`end=` controls what is written after the last argument. It defaults to `"\n"`. Setting it to `""` gives you `Console.Write` behaviour.

```csharp
// C#
Console.Write("Loading");
Console.Write("...");
Console.WriteLine(" done");
```

In [ ]:
print("Loading", end="")
print("...", end="")
print(" done")

for i in range(5):
    print(i, end=" ")       # all on one line
print()                     # finish the line

## `file` — writing to stderr (or anything)

By default `print()` writes to `sys.stdout`. Pass `file=` to write somewhere else — typically `sys.stderr` for errors and diagnostics (C#: `Console.Error`). In Jupyter, stderr output is shown with a red/pink background.

```csharp
// C#
Console.Error.WriteLine("Warning: config file not found");
```

In [ ]:
import sys

print("normal output")
print("Warning: config file not found", file=sys.stderr)

`file=` accepts any object with a `write()` method — an open file, or an in-memory `io.StringIO` (C#: `StringWriter`):

In [ ]:
import io

buffer = io.StringIO()
print("line one", file=buffer)
print("line two", file=buffer)

print(repr(buffer.getvalue()))

## `flush`

Standard output is buffered. When writing to a terminal it is usually flushed at each newline, but when output is piped or redirected it may sit in a buffer. `flush=True` forces it out immediately — useful for progress messages in long-running scripts.

In [ ]:
import time

for step in range(3):
    print(f"step {step} ", end="", flush=True)   # appears immediately, even without a newline
    time.sleep(0.1)
print("finished")

## f-strings for building output

The idiomatic way to mix text and values is an **f-string** — a string literal prefixed with `f` whose `{...}` placeholders are evaluated as expressions. It is the direct equivalent of C# string interpolation `$"..."`.

```csharp
// C#
var name = "alice";
var items = 3;
Console.WriteLine($"{name} has {items} items");
Console.WriteLine($"Price: {9.5:F2}");
```

In [ ]:
name = "alice"
items = 3
price = 9.5

print(f"{name} has {items} items")
print(f"Next year: {items + 1} items")      # any expression works
print(f"Price: {price:.2f}")                # format spec, like C#'s :F2
print(f"{name=}, {items=}")                 # debug form: prints name='alice', items=3

Format specifiers, alignment, padding, `str.format()`, `%`-formatting and the new template strings are covered in depth later *(see Data Types › 003 · String Formatting)*.

## How objects are printed: `str()` vs `repr()`

`print()` uses `str(obj)` — the human-friendly form (C#: `ToString()`). The REPL and notebook display (and containers such as lists) use `repr(obj)` — the unambiguous, developer-facing form. The difference is most visible with strings:

In [ ]:
text = "hello"

print(text)          # str():  hello
print(repr(text))    # repr(): 'hello'  — quotes show it's a string
print([text])        # a list shows its items with repr()
text                 # notebook display uses repr()

You control both forms for your own classes with `__str__` and `__repr__` *(see Classes & OOP › 007 · Dunder Methods)*.

## Reading input with `input()`

`input(prompt)` writes the prompt (without a newline), waits for the user to press Enter, and returns the line as a `str` — the trailing newline is stripped. It is `Console.Write(prompt)` + `Console.ReadLine()` in one call.

```csharp
// C#
Console.Write("What is your name? ");
string? name = Console.ReadLine();       // null at end of input
Console.WriteLine($"Hello, {name}!");
```

```python
# Python — not executed here: notebooks run non-interactively in this course
name = input("What is your name? ")
print(f"Hello, {name}!")
```

Differences from `Console.ReadLine()`:

- `input()` never returns `None`; at end of input (Ctrl+D / Ctrl+Z, or a closed pipe) it raises `EOFError` instead of returning `null`
- the result is **always** a `str` — even if the user typed digits

## Converting input: `int()` and `float()`

Because `input()` returns text, you convert it explicitly — `int(s)` is the counterpart of `int.Parse(s)`. Invalid text raises `ValueError` (C#: `FormatException`). There is no `TryParse`; the Pythonic approach is to try the conversion and catch the exception.

```csharp
// C#
if (int.TryParse(Console.ReadLine(), out var age))
    Console.WriteLine($"Next year you'll be {age + 1}");
else
    Console.WriteLine("That's not a number");
```

```python
# Python — the interactive version
raw = input("How old are you? ")
```

We simulate the user's typed text with plain strings so the code can run here:

In [ ]:
for raw in ["41", "  7 \n", "forty"]:      # what input() might return
    try:
        age = int(raw)                     # int() tolerates surrounding whitespace
        print(f"Next year you'll be {age + 1}")
    except ValueError as e:
        print(f"{type(e).__name__}: {e}")

In [ ]:
height = float("1.82")       # float.Parse
numbers = [int(part) for part in "3 5 8".split()]   # several values on one line

print(height, numbers, sum(numbers))

⚠️ **Gotcha:** forgetting to convert is a classic bug — `input()` returns a `str`, and `+` on two strings concatenates:

In [ ]:
a = "2"    # pretend these came from input()
b = "3"
print(a + b)             # '23' — string concatenation, not 5!
print(int(a) + int(b))   # 5

💡 For real command-line tools, prefer arguments over interactive prompts: `sys.argv` (C#: `args` in `Main`) or the `argparse` module for proper option parsing.

## Pretty-printing with `pprint`

Printing a large nested structure puts it all on one long line. The `pprint` module ("pretty print") formats it across lines with indentation — handy when exploring data, much like inspecting an object in the debugger. Dictionaries are shown with keys sorted by default.

In [ ]:
from pprint import pprint

order = {
    "id": 1001,
    "customer": {"name": "alice", "tier": "gold"},
    "items": [
        {"product": "keyboard", "qty": 1, "price": 49.99},
        {"product": "mouse", "qty": 2, "price": 19.99},
    ],
    "status": "shipped",
}

print(order)          # one long line
print()
pprint(order, width=60, sort_dicts=False)   # keep insertion order

In [ ]:
from pprint import pformat

text = pformat(list(range(30)), width=40, compact=True)   # returns a str instead of printing
print(text)

## Summary

| Task | C# | Python |
|---|---|---|
| Write a line | `Console.WriteLine(x)` | `print(x)` |
| Write without newline | `Console.Write(x)` | `print(x, end="")` |
| Several values | `string.Join(" ", a, b)` | `print(a, b)` |
| Custom separator | `string.Join(", ", ...)` | `print(a, b, sep=", ")` |
| Write to stderr | `Console.Error.WriteLine(x)` | `print(x, file=sys.stderr)` |
| Flush | `Console.Out.Flush()` | `print(x, flush=True)` |
| Interpolation | `$"Hi {name}"` | `f"Hi {name}"` |
| Read a line | `Console.ReadLine()` | `input()` |
| Parse a number | `int.Parse(s)` / `TryParse` | `int(s)` + `try/except ValueError` |
| Pretty-print | debugger / JSON serializer | `pprint(obj)` |

## Exercises

1. Print the numbers 1 to 5 on a single line separated by ` | ` (i.e. `1 | 2 | 3 | 4 | 5`) using a single `print()` call. *Hint:* unpack a range with `*`.
2. Write a function `parse_age(raw)` that returns the integer age, or `None` if `raw` is not a valid whole number. Test it with `"30"`, `" 12 "` and `"abc"`.
3. Print a progress bar `[#####]` one `#` at a time on the same line, then print `done` on the next line.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1 — *range(...) passes each number as a separate argument
print(*range(1, 6), sep=" | ")

# 2 — the Pythonic TryParse
def parse_age(raw):
    try:
        return int(raw)
    except ValueError:
        return None

print(parse_age("30"), parse_age(" 12 "), parse_age("abc"))   # 30 12 None

# 3
print("[", end="")
for _ in range(5):
    print("#", end="", flush=True)
print("]")
print("done")
```

</details>